# SIH26012 — AI-Based Automated Urban Parcel Mapping
## Phase 3: Progressive Multiscale Generator (PMG) Cadastral Segmentation Model

This notebook trains and evaluates the **Progressive Multiscale Generator (PMG)** segmentation architecture on 512×512 CadastreVision aerial imagery patches using combined Focal + Dice loss with PyTorch AMP mixed precision in Google Colab Pro.

### Key Architectural Features:
- **Multi-Scale Feature Branches**: Processes hierarchical encoder features (32, 64, 128, 256, 512) via multi-kernel receptive field blocks (1×1, 3×3, 5×5 equivalent dilated convolutions).
- **Progressive Coarse-to-Fine Fusion**: Aggregates higher-level global cadastral context down to fine boundary details.
- **Residual Skip Injection**: Enriches U-Net decoder pathways with multiscale context maps.

In [ ]:
# 1. Install & Import Dependencies
!pip install -q rasterio geopandas shapely pyogrio opencv-python-headless matplotlib pillow pyyaml tqdm

import os, sys, time, json, random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch Version: {torch.__version__}")

In [ ]:
# 2. GPU Detection & Memory Safety
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"))
print(f"Active Compute Device: {device}")

if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.backends.cudnn.benchmark = True

In [ ]:
# 3. Model Architecture: PMG + Residual U-Net
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, residual=True):
        super().__init__()
        self.residual = residual
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ) if (residual and in_channels != out_channels) else nn.Identity()

    def forward(self, x):
        res = self.shortcut(x) if self.residual else 0
        return F.relu(self.conv(x) + res, inplace=True)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.mpconv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    def forward(self, x):
        return self.mpconv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        diff_y = x2.size()[2] - x1.size()[2]
        diff_x = x2.size()[3] - x1.size()[3]
        if diff_y > 0 or diff_x > 0:
            x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MultiscaleFeatureBranch(nn.Module):
    def __init__(self, in_channels, out_channels, dilation_rates=[1, 2, 4]):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True)
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=3, padding=dilation_rates[0], dilation=dilation_rates[0], bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True)
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=3, padding=dilation_rates[1], dilation=dilation_rates[1], bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True)
        )
        self.branch4 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=3, padding=dilation_rates[2], dilation=dilation_rates[2], bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True)
        )
        self.out_conv = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        cat = torch.cat([b1, b2, b3, b4], dim=1)
        return self.out_conv(cat)

class PMGModule(nn.Module):
    def __init__(self, in_channels_list=[64, 128, 256, 512], out_channels=128):
        super().__init__()
        self.num_scales = len(in_channels_list)
        self.branches = nn.ModuleList([
            MultiscaleFeatureBranch(in_ch, out_channels) for in_ch in in_channels_list
        ])
    def forward(self, features):
        projected = [self.branches[i](features[i]) for i in range(self.num_scales)]
        refined = [None] * self.num_scales
        refined[-1] = projected[-1]
        for i in range(self.num_scales - 2, -1, -1):
            coarser_up = F.interpolate(refined[i + 1], size=projected[i].shape[2:], mode="bilinear", align_corners=False)
            refined[i] = projected[i] + coarser_up
        return refined

class CadastreUNetPMG(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[32, 64, 128, 256, 512], pmg_channels=128):
        super().__init__()
        f = features
        self.inc = DoubleConv(in_channels, f[0])
        self.down1 = Down(f[0], f[1])
        self.down2 = Down(f[1], f[2])
        self.down3 = Down(f[2], f[3])
        self.down4 = Down(f[3], f[4])
        self.pmg = PMGModule(in_channels_list=[f[1], f[2], f[3], f[4]], out_channels=pmg_channels)
        self.pmg_adapters = nn.ModuleList([
            nn.Conv2d(pmg_channels, f[i], kernel_size=1) for i in [1, 2, 3, 4]
        ])
        self.up1 = Up(f[4], f[3])
        self.up2 = Up(f[3], f[2])
        self.up3 = Up(f[2], f[1])
        self.up4 = Up(f[1], f[0])
        self.outc = nn.Conv2d(f[0], out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        pmg_feats = self.pmg([x2, x3, x4, x5])
        x2 = x2 + self.pmg_adapters[0](pmg_feats[0])
        x3 = x3 + self.pmg_adapters[1](pmg_feats[1])
        x4 = x4 + self.pmg_adapters[2](pmg_feats[2])
        x5 = x5 + self.pmg_adapters[3](pmg_feats[3])
        d = self.up1(x5, x4)
        d = self.up2(d, x3)
        d = self.up3(d, x2)
        d = self.up4(d, x1)
        return self.outc(d)

model = CadastreUNetPMG().to(device)
print(f"PMG Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 4. Loss Function: Focal Loss + Dice Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt = probs * targets + (1 - probs) * (1 - targets)
        focal_weight = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = focal_weight * ((1 - pt) ** self.gamma) * bce
        return loss.mean()

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum(dim=(2, 3))
        cardinality = (probs + targets).sum(dim=(2, 3))
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return (1.0 - dice).mean()

class CombinedFocalDiceLoss(nn.Module):
    def __init__(self, focal_w=0.5, dice_w=0.5):
        super().__init__()
        self.focal = FocalLoss()
        self.dice = DiceLoss()
        self.focal_w, self.dice_w = focal_w, dice_w
    def forward(self, logits, targets):
        return self.focal_w * self.focal(logits, targets) + self.dice_w * self.dice(logits, targets)

criterion = CombinedFocalDiceLoss()

## 5. Training Loop (15 Epochs with AMP & Early Stopping)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
print("Ready to train PMG model with Fast 15-Epoch CosineAnnealingLR Schedule.")